# 02 — Analyse exploratoire (données réelles)
## Démographie, santé & conditions de vie au Sénégal

Questions : Où en est la **transition démographique** ? Comment ont reculé la
**mortalité infantile** et la **pauvreté** ? Quelles **disparités régionales**
(fécondité, électricité, éducation) ? Quels liens entre éducation et fécondité ?

In [1]:

import os, warnings, json, re, unicodedata
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.figsize": (11, 5), "figure.dpi": 110, "axes.titlesize": 13})

PROJ = os.getcwd()
if not os.path.isdir(os.path.join(PROJ, "data")):
    PROJ = os.path.dirname(PROJ)
RAW = os.path.join(PROJ, "data", "raw"); PROC = os.path.join(PROJ, "data", "processed")
GEO = os.path.join(PROJ, "data", "geo"); FIG = os.path.join(PROJ, "reports", "figures")
MODELS = os.path.join(PROJ, "models")
for d in (PROC, FIG, MODELS): os.makedirs(d, exist_ok=True)

def deacc(s):
    return "".join(c for c in unicodedata.normalize("NFD", str(s)) if unicodedata.category(c) != "Mn")
print("Racine projet :", PROJ)


Racine projet : C:\projet\senegal-demographie


In [2]:

nat = pd.read_csv(os.path.join(PROC, "dhs_national_wide.csv"))
sub = pd.read_csv(os.path.join(PROC, "dhs_subnational.csv"))
subw = pd.read_csv(os.path.join(PROC, "dhs_subnational_wide.csv"))
wb = pd.read_csv(os.path.join(PROC, "worldbank_wide.csv"))
dim = pd.read_csv(os.path.join(PROC, "dim_indicator.csv"))
LAB = dict(zip(dim["code"], dim["indicateur"]))
print("OK |", nat.shape, subw.shape, wb.shape)


OK | (16, 11) (14, 12) (65, 12)


### 1. Transition démographique : fécondité (EDS) & espérance de vie (BM)

In [3]:

fig, ax1 = plt.subplots()
t = nat.dropna(subset=["FE_FRTR_W_TFR"])
ax1.plot(t["annee"], t["FE_FRTR_W_TFR"], "-o", color="#1f4e79", lw=2, label="Fécondité (enfants/femme)")
ax1.axhline(2.1, color="grey", ls="--", lw=.8); ax1.text(t["annee"].min(), 2.25, "Seuil de renouvellement (2,1)", fontsize=8, color="grey")
ax1.set_ylabel("Indice de fécondité", color="#1f4e79")
ax2 = ax1.twinx()
le = wb.dropna(subset=["SP.DYN.LE00.IN"])
ax2.plot(le["annee"], le["SP.DYN.LE00.IN"], color="#27ae60", lw=1.6, label="Espérance de vie (ans)")
ax2.set_ylabel("Espérance de vie (ans)", color="#27ae60")
ax1.set_title("Transition démographique au Sénégal (1986–2023)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "01_transition.png"), bbox_inches="tight")
plt.close(fig); print("→ 01_transition.png")


→ 01_transition.png


### 2. Recul de la mortalité des enfants (EDS)

In [4]:

fig, ax = plt.subplots()
for code, col, lab in [("CM_ECMR_C_U5M","#c0392b","Moins de 5 ans"),
                       ("CM_ECMR_C_IMR","#e67e22","Infantile (<1 an)"),
                       ("CM_ECMR_C_NNR","#8e44ad","Néonatale")]:
    g = nat.dropna(subset=[code])
    ax.plot(g["annee"], g[code], "-o", color=col, lw=1.7, label=lab)
ax.set_ylabel("Décès pour 1 000 naissances vivantes"); ax.legend()
ax.set_title("Recul de la mortalité des enfants au Sénégal (EDS)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "02_mortalite.png"), bbox_inches="tight")
plt.close(fig); print("→ 02_mortalite.png")


→ 02_mortalite.png


### 3. Population, structure par âge et urbanisation (Banque mondiale)

In [5]:

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
# population + croissance
p = wb.dropna(subset=["SP.POP.TOTL"])
axes[0].plot(p["annee"], p["SP.POP.TOTL"]/1e6, color="#1f4e79", lw=2)
axes[0].set_title("Population totale (millions)"); axes[0].set_ylabel("Millions")
# structure par âge (aire empilée)
a = wb.dropna(subset=["SP.POP.0014.TO.ZS"])
axes[1].stackplot(a["annee"], a["SP.POP.0014.TO.ZS"], a["SP.POP.1564.TO.ZS"], a["SP.POP.65UP.TO.ZS"],
                  labels=["0-14 ans","15-64 ans","65+ ans"], colors=["#74add1","#fdae61","#d73027"])
axes[1].legend(loc="center left", fontsize=8); axes[1].set_title("Structure par âge (%)"); axes[1].set_ylim(0,100)
fig.suptitle("Dynamique de population au Sénégal", y=1.02)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "03_population_age.png"), bbox_inches="tight")
plt.close(fig); print("→ 03_population_age.png | jeunes <15 ans :",
      round(a['SP.POP.0014.TO.ZS'].iloc[-1],1), "%")


→ 03_population_age.png | jeunes <15 ans : 38.2 %


### 4. Pauvreté & inégalités (Banque mondiale)

In [6]:

fig, ax1 = plt.subplots()
pov = wb.dropna(subset=["SI.POV.NAHC"])
ax1.plot(pov["annee"], pov["SI.POV.NAHC"], "-o", color="#c0392b", lw=2, label="Pauvreté nationale (%)")
ax1.set_ylabel("Taux de pauvreté national (%)", color="#c0392b")
gini = wb.dropna(subset=["SI.POV.GINI"])
ax2 = ax1.twinx()
ax2.plot(gini["annee"], gini["SI.POV.GINI"], "-s", color="#1f4e79", lw=1.5, label="Indice de Gini")
ax2.set_ylabel("Indice de Gini", color="#1f4e79")
ax1.set_title("Pauvreté et inégalités au Sénégal")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "04_pauvrete.png"), bbox_inches="tight")
plt.close(fig); print("→ 04_pauvrete.png")


→ 04_pauvrete.png


### 5. Disparités régionales (EDS 2023)

In [7]:

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
plots = [("FE_FRTR_W_TFR","Fécondité (enfants/femme)","flare", False),
         ("HC_ELEC_H_ELC","Accès électricité (%)","crest", True),
         ("ED_LITR_W_LIT","Alphabétisation femmes (%)","viridis", True)]
for ax,(code,title,cmap,asc) in zip(axes, plots):
    s = subw[["region",code]].dropna().sort_values(code, ascending=asc)
    sns.barplot(data=s, y="region", x=code, ax=ax, palette=cmap)
    ax.set_title(title); ax.set_xlabel(""); ax.set_ylabel("")
fig.suptitle("Disparités régionales — EDS 2023", y=1.03)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "05_regions_bars.png"), bbox_inches="tight")
plt.close(fig); print("→ 05_regions_bars.png")


→ 05_regions_bars.png


### 6. Cartes choroplèthes régionales (EDS 2023)

In [8]:

from matplotlib.patches import Polygon as MplPoly
import matplotlib.colors as mcolors, matplotlib.cm as cm
geo = json.load(open(os.path.join(GEO, "senegal_regions.geojson"), encoding="utf-8"))
def rings(g): return [g["coordinates"]] if g["type"]=="Polygon" else g["coordinates"]

def choro(ax, code, title, cmap, fmt="{:.0f}"):
    vmap = dict(zip(subw["region_geo"], subw[code]))
    vals = [v for v in vmap.values() if pd.notna(v)]
    norm = mcolors.Normalize(min(vals), max(vals)); sm = cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    for feat in geo["features"]:
        nm = feat["properties"]["shapeName"]; val = vmap.get(nm)
        color = sm.to_rgba(val) if val is not None and pd.notna(val) else "#eee"
        big=-1; c=None
        for poly in rings(feat["geometry"]):
            ext=np.array(poly[0]); ax.add_patch(MplPoly(ext, closed=True, facecolor=color, edgecolor="white", lw=.5))
            ar=abs(np.sum(ext[:,0]*np.roll(ext[:,1],1)-np.roll(ext[:,0],1)*ext[:,1]))/2
            if ar>big: big,c=ar,(ext[:,0].mean(),ext[:,1].mean())
        if c is not None and val is not None and pd.notna(val):
            ax.annotate(fmt.format(val), c, ha="center", va="center", fontsize=7, weight="bold")
    ax.autoscale(); ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title, fontsize=12)
    cb=plt.colorbar(sm, ax=ax, shrink=.55);
fig, axes = plt.subplots(1, 3, figsize=(16, 6))
choro(axes[0], "FE_FRTR_W_TFR", "Fécondité (enfants/femme)", "Reds", "{:.1f}")
choro(axes[1], "HC_ELEC_H_ELC", "Accès à l'électricité (%)", "Greens")
choro(axes[2], "CN_NUTS_C_HA2", "Malnutrition chronique (%)", "OrRd")
fig.suptitle("Cartes régionales — EDS 2023 (données réelles)", y=1.02)
fig.tight_layout(); fig.savefig(os.path.join(FIG, "06_cartes_regionales.png"), bbox_inches="tight", dpi=120)
plt.close(fig); print("→ 06_cartes_regionales.png")


→ 06_cartes_regionales.png


### 7. Lien éducation ↔ fécondité (régions, 2023)

In [9]:

s = subw.dropna(subset=["ED_LITR_W_LIT","FE_FRTR_W_TFR"])
corr = s["ED_LITR_W_LIT"].corr(s["FE_FRTR_W_TFR"])
fig, ax = plt.subplots()
ax.scatter(s["ED_LITR_W_LIT"], s["FE_FRTR_W_TFR"], s=60, color="#1f4e79")
for _, r in s.iterrows():
    ax.annotate(r["region"], (r["ED_LITR_W_LIT"], r["FE_FRTR_W_TFR"]), fontsize=7, xytext=(3,3), textcoords="offset points")
z = np.polyfit(s["ED_LITR_W_LIT"], s["FE_FRTR_W_TFR"], 1)
xs = np.linspace(s["ED_LITR_W_LIT"].min(), s["ED_LITR_W_LIT"].max(), 50)
ax.plot(xs, np.polyval(z, xs), color="#c0392b", ls="--", label=f"corr = {corr:.2f}")
ax.set_xlabel("Alphabétisation des femmes (%)"); ax.set_ylabel("Fécondité (enfants/femme)")
ax.legend(); ax.set_title("Plus d'éducation des femmes ↔ moins d'enfants (régions, 2023)")
fig.tight_layout(); fig.savefig(os.path.join(FIG, "07_education_fecondite.png"), bbox_inches="tight")
plt.close(fig); print("→ 07_education_fecondite.png | corr =", round(corr,2))


→ 07_education_fecondite.png | corr = -0.64


### Synthèse EDA
- **Transition démographique** en cours : fécondité de **6,4 (1986) → 4,0 (2023)**,
  espérance de vie en forte hausse, mais population encore très **jeune**.
- **Mortalité des enfants** fortement réduite depuis les années 1990.
- **Pauvreté** en recul mais encore élevée ; inégalités modérées (Gini ~36).
- **Fortes disparités régionales** : Dakar/ouest urbanisés (faible fécondité, fort
  accès aux services) vs régions du sud/est (fécondité élevée, accès plus faible).
- L'**éducation des femmes** est nettement corrélée à une **fécondité plus basse**.
